# Resposta del consum elèctric de Barcelona a la temperatura
### Notebook v0.13 · article 1 · versió depurada (només el que és publicable)

**Tesi:** *la ciutat observada a través de les seves respostes* (Pau Buchaca, ETSAB-UPC). Peça P1.

**Què afirma l'article (i on es comprova en aquest notebook)**

| # | Resultat | Secció |
|---|---|---|
| R1 | La ciutat respon a la calor amb ~6 % de consum residencial per grau, amb memòria d'1-2 dies, i la resposta lineal **prediu les onades** | 6 |
| R2 | La resposta **creix**: del 2021 al 2024 gairebé es duplica, i no és un artefacte de composició, de cobertura, de llindar, d'humitat ni de preu | 7 |
| R3 | El creixement és un canvi d'**amplitud**: cada estiu té la mateixa corba de resposta multiplicada per un factor (model de llindars) | 8 |
| R4 | La resposta que creix és la de la **calor de nit**, a totes les franges horàries; la resposta a la calor de dia no creix | 9 |
| R5 | El fred: resposta estable; la V es torna **menys simètrica** | 10 |
| R6 | **No hi ha histèresi** estreta: el sistema és una memòria lineal (*crossover*, no transició) | 11 |
| — | Sensibilitat de la calor antropogènica $\partial Q_F/\partial T$ (mesurada) i ordre de magnitud del bucle $G$ (no verificat) | 12 |

**Model**
$$\ln E(d) = \sum_{k=0}^{3} \chi(k)\,\mathrm{CDD}(d-k) + \text{any} + \text{setmana ISO} + \text{dia de la setmana} + \text{festiu} + \varepsilon,\qquad \mathrm{CDD}=\max(T-T_c,0)$$
$\chi^0=\sum_k\chi(k)$ és el % de consum de més per cada grau-dia. Errors per clústers de setmana.

**Què NO hi ha (i on és):** tot el que és per codi postal, mapes, illa de calor per satèl·lit, renda, turisme i hotels. Les alertes de la font ho fan fràgil o ja s'ha descartat. És tot a l'arxiu `resposta_calor_bcn_v12.ipynb`.

**Com usar-lo:** `Entorn d'execució → Executa-ho tot`. Totes les baixades tenen memòria cau a `dades_tesi`. El notebook **s'atura** si una alerta coneguda no queda tractada. Al final desa `dades_per_claude_v13.zip` al Drive.

**Convenció:** cada secció té una cel·la de text (què fa, per què, què cal mirar) i el codi crida funcions del paquet `consum_temperatura` (repositori `power_temperature_urban`). Totes estan provades amb dades sintètiques on la resposta és coneguda.

## 1. Instal·lació

Instal·la el paquet de la tesi directament de GitHub. Si el repositori canvia, tornar a executar aquesta cel·la porta la versió nova.

In [ ]:
!pip install -q holidays "git+https://github.com/PauBuchacaMartin/power_temperature_urban@main"

import os, json, zipfile, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import holidays
from scipy import stats
import consum_temperatura as ct
from consum_temperatura import figstyle as fs

fs.apply_style()
pd.set_option("display.width", 160)
print(f"[OK] consum_temperatura {ct.__version__}")

## 2. Especificació congelada

**Què fa:** fixa **abans de mirar els resultats** totes les decisions de l'anàlisi. Si algun resultat depèn d'una decisió presa després de veure les dades, ho diu explícitament (marca *post hoc*).

**Per què:** és la regla que protegeix l'article de triar l'especificació que dona el resultat desitjat.

In [ ]:
SPEC = {
    # Dades
    "carpeta": "/content/drive/MyDrive/dades_tesi",
    "anys": [2021, 2022, 2023, 2024, 2025],
    "lat_lon": (41.39, 2.17),                  # ERA5 (Open-Meteo), una sola sèrie per a la ciutat
    # Model principal (R1, R2)
    "mesos_estiu": [6, 7, 8, 9],
    "retards": 3,                              # memòria: dies 0..3
    "tc_candidats": list(np.arange(18.0, 25.5, 0.5)),
    "onada_percentil": 0.95, "onada_dies": 3,  # onada: Tmax > P95 de l'estiu durant >= 3 dies
    "any_ref": 2021, "any_fi": 2024,           # creixement principal (el 2025 té qualitat inferior: alerta 1)
    # Dia–nit (R4)
    "tc_dia": 27.0, "tc_nit": 20.0,            # T 12-18 h i T 0-6 h (fixats a la v0.10)
    "anys_franges": [2021, 2022, 2023, 2024],  # el 2025 és gairebé tot del règim B (alerta 2)
    # Fred (R5)
    "hiverns": [2021, 2022, 2023, 2024],       # hivern 2021 = nov. 2021 - març 2022
    "mesos_hivern": [11, 12, 1, 2, 3],         # especificació inicial (v0.12)
    "mesos_hivern_cor": [12, 1, 2],            # POST HOC (24/10/2026): triat després de veure que nov. i març dominaven
    "th_candidats": list(np.arange(10.0, 19.5, 0.5)),
    # Qualitat de dades
    "anomalia_k": 6.0, "bloc_llindar": 0.12, "bloc_dies": 7,
    # Calor antropogènica
    "area_km2": 101.0,                         # àrea dels 42 CP (reconstrucció per Voronoi d'adreces, v0.2.3)
    "h_eff": (10.0, 50.0),                     # W m-2 K-1 · NO VERIFICAT: només per a un ordre de magnitud de G
}
ANYS, LAGS = SPEC["anys"], SPEC["retards"]
A0, A1 = SPEC["any_ref"], SPEC["any_fi"]
os.makedirs("figures_v13", exist_ok=True)
print(json.dumps({k: v for k, v in SPEC.items() if k != "tc_candidats" and k != "th_candidats"}, indent=1, ensure_ascii=False))

## 3. Dades

**Què fa:** llegeix els CSV de consum elèctric d'Open Data BCN (origen Datadis) de la carpeta `dades_tesi`, la temperatura ERA5 diària i horària, la humitat i el preu (de la memòria cau), i els festius de Catalunya.

**Què cal mirar:** que el consum anual del residencial sigui d'uns 2.000 GWh. El portal diu que els valors són MWh, però són **kWh**: amb MWh, el residencial de Barcelona seria ~2.000 TWh/any, gairebé deu vegades tot el consum d'Espanya. La cel·la ho comprova sola.

In [ ]:
from pathlib import Path
if SPEC["carpeta"].startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
CARPETA = SPEC["carpeta"]

# --- Consum: un CSV per any, tal com el dona el portal ---
trossos = []
for f in sorted(Path(CARPETA).glob("*.csv")):
    if "consum_electricitat" in f.name.lower() and any(str(a) in f.name for a in ANYS):
        print("  llegint", f.name)
        trossos.append(ct.daily_consumption(pd.read_csv(f)))
assert trossos, f"No hi ha CSV de consum a {CARPETA} (han de contenir 'consum_electricitat' i l'any al nom)"
df = pd.concat(trossos, ignore_index=True)
df = df[df["data"].dt.year.isin(ANYS)]
assert len(df), "Els CSV s'han llegit però no en surt cap fila: revisa els noms de les columnes"
print(f"[OK] Consum: {len(df):,} files · {df['data'].min().date()} → {df['data'].max().date()} · "
      f"{df['cp'].nunique()} CP · sectors {sorted(df['sector'].unique())}")

# Comprovació d'unitats (alerta 6): residencial anual entre 1.000 i 4.000 GWh si els valors són kWh
gwh = df[df["sector"] == "Residencial"].groupby(df["data"].dt.year)["kwh"].sum() / 1e6
print("Residencial anual (GWh, si els valors són kWh):", gwh.round(0).to_dict())
assert gwh.loc[[a for a in gwh.index if a <= 2024]].between(1000, 4000).all(), "Unitats inesperades: revisa el CSV"

# --- Temperatura diària i horària (ERA5, Open-Meteo), amb memòria cau ---
lat, lon = SPEC["lat_lon"]
fi = df["data"].max().strftime("%Y-%m-%d")
temp = ct.from_cache(CARPETA, "temperatura.csv", lambda: ct.era5_daily(lat, lon, f"{ANYS[0]}-01-01", fi), until=fi)
temp = temp[["data", "tmax", "tmean"]].dropna().sort_values("data").reset_index(drop=True)
hor = ct.from_cache(CARPETA, "temp_horaria.csv", lambda: ct.era5_hourly(lat, lon, f"{ANYS[0]}-01-01", fi), date_col="hora")
hor = hor.rename(columns={"temperature_2m": "temp"})[["hora", "temp"]].dropna()

# --- Humitat i preu: només de la memòria cau (els va baixar la v0.11); si no hi són, les proves se salten ---
def opcional(nom):
    ruta = f"{CARPETA}/{nom}"
    return pd.read_csv(ruta, parse_dates=["data"]) if os.path.exists(ruta) else None
humitat, preus = opcional("humitat_era5.csv"), opcional("preus_electricitat.csv")
print("Humitat:", "sí" if humitat is not None else "NO", "· Preu:", "sí" if preus is not None else "NO")

FESTIUS = sorted(pd.to_datetime(list(holidays.country_holidays("ES", subdiv="CT", years=ANYS).keys())))
temp["onada"] = ct.heatwaves(temp, SPEC["onada_percentil"], SPEC["onada_dies"]).values
ONADES = set(temp.loc[temp["onada"], "data"])
print(f"[OK] Temperatura: {len(temp)} dies · dies d'onada: {len(ONADES)}")

## 4. Qualitat de les dades i registre d'alertes

**Què fa, en quatre passos:**
1. **Dies complets:** un dia entra només si hi són tots els CP actius aquell trimestre (si no, la suma de la ciutat surt baixa).
2. **Auditoria per sectors:** dibuixa el consum diari de tota la ciutat per sector. Aquí es veuen les alertes 3 i 4.
3. **Anomalies condicionades a la temperatura**, per a cada sèrie (residencial, serveis, total). Un model flexible (graus d'1 °C + any + setmana + dia + festiu) explica la V, i el que queda fora és sospitós:
   - **dies sols** amb un residu > 6 σ, excepte els dies de calor o fred extrems, que mai s'exclouen (són el senyal);
   - **blocs negatius**: una mediana mòbil de 15 dies per sota de −12 % durant ≥ 7 dies, que vol dir que falten subministraments.

   Els blocs positius es mostren però no s'exclouen: dues setmanes de calor persistent (agost del 2024) apugen el consum per sobre del model diari.
4. **Règim de les franges horàries** (alerta 2).

**Comprovacions automàtiques:** el notebook s'atura si el forat dels serveis del 2023 no queda exclòs o si s'exclou algun dia d'onada del residencial.

**Què cal mirar:** la taula-registre del final. Cada alerta hi té el seu efecte i la seva solució.

In [ ]:
# 4.1 Dies complets (amb els CP del residencial)
res_cp = df[df["sector"] == "Residencial"].groupby(["data", "cp"], as_index=False)["kwh"].sum()
COMPLETS = set(ct.complete_days(res_cp))
print(f"Dies complets: {len(COMPLETS)} de {res_cp['data'].nunique()}")

# 4.2 Auditoria per sectors (alertes 3 i 4)
sect = df.groupby(["data", "sector"])["kwh"].sum().unstack().fillna(0) / 1e3
fig, axs = plt.subplots(len(sect.columns) + 1, 1, figsize=(12, 2.1 * (len(sect.columns) + 1)), sharex=True)
for ax, col in zip(axs, list(sect.columns) + ["Total"]):
    y = sect.sum(axis=1) if col == "Total" else sect[col]
    ax.plot(y.index, y, lw=0.5, color=fs.CAT[0]); ax.plot(y.index, y.rolling(28, center=True).median(), lw=1.2, color=fs.INK)
    ax.set_ylabel(f"{col}\nMWh/dia", fontsize=8)
axs[0].set_title("Consum diari de tota la ciutat per sector (línia negra: mediana de 28 dies)")
plt.tight_layout(); plt.savefig("figures_v13/auditoria_sectors.png"); plt.show()
print("Quota de cada sector (%) per trimestre:")
print(ct.sector_shares(df).round(1).to_string())

In [ ]:
# 4.3 Anomalies condicionades a la temperatura, per a cada sèrie
EXTREMS = ct.extreme_days(temp)
kw_an = dict(k=SPEC["anomalia_k"], block_threshold=SPEC["bloc_llindar"], block_min_days=SPEC["bloc_dies"], protect=EXTREMS)
SERIES, DIES_BONS, ANOMALIES = {}, {}, {}
for nom, sec in [("Residencial", "Residencial"), ("Serveis", "Serveis"), ("Total", None)]:
    s = ct.city_series(df, sector=sec)
    bons, an = ct.clean_days(s, temp, FESTIUS, complete=COMPLETS, **kw_an)
    SERIES[nom], DIES_BONS[nom], ANOMALIES[nom] = s, bons, an
    print(f"\n{nom}: {len(bons)} dies bons · {int(an['single'].sum())} dies sols i {int(an['block'].sum())} dies de bloc exclosos")
    blocs = ct.anomaly_blocks(an)
    if len(blocs): print("  Blocs negatius (exclosos):\n" + blocs.to_string(index=False))
    pos = ct.anomaly_blocks(an, "block_pos")
    if len(pos): print("  Blocs positius (NO exclosos):\n" + pos.to_string(index=False))
    if an["protected"].any(): print("  Dies extrems marcats però no exclosos:", [str(d.date()) for d in an.index[an["protected"]]])

# Sèries netes per a l'anàlisi (les mateixes en totes les seccions)
NET = {nom: s[s["data"].isin(DIES_BONS[nom])].reset_index(drop=True) for nom, s in SERIES.items()}

# Comprovacions automàtiques: les alertes conegudes han de quedar tractades
forat = pd.date_range("2023-01-15", "2023-03-15")
assert not NET["Serveis"]["data"].isin(forat).any(), "ALERTA 3 no tractada: el forat dels serveis del 2023 és a l'anàlisi"
assert not (set(SERIES["Residencial"]["data"]) - DIES_BONS["Residencial"]) & ONADES & COMPLETS, "S'ha exclòs un dia d'onada del residencial"
print("\n[OK] Alerta 3 tractada (forat dels serveis exclòs) i cap dia d'onada del residencial exclòs")
pd.concat({n: a[a["single"] | a["block"]] for n, a in ANOMALIES.items()}).to_csv("dies_exclosos_v13.csv")

In [ ]:
# 4.4 Règim de les franges horàries (alerta 2)
FRANGES = sorted(t for t in df["tram"].dropna().unique() if "consta" not in str(t).lower())
NOM_F = {t: f"{int(str(t).split(':')[0][-2:]):02d}-{int(str(t).split(':')[0][-2:]) + 6:02d}" for t in FRANGES}
pb = (df[(df["sector"] == "Residencial") & df["tram"].isin(FRANGES)]
      .pivot_table(index="data", columns="tram", values="kwh", aggfunc="sum").rename(columns=NOM_F))
anom_f, REGIM_B = ct.band_regime(pb, evening="18-24", morning="06-12")
print("Dies del règim B (repartiment entre franges canviat) per any:",
      pd.Series([d.year for d in REGIM_B]).value_counts().sort_index().to_dict())

# 4.5 Registre d'alertes
REGISTRE = pd.DataFrame([
    ["1", "Recodificació de CP (tardor 2023, juliol 2025)", "per CP; no a la ciutat", "tot a escala de ciutat; creixement 2021→2024; el 2025 només com a robustesa"],
    ["2", "Canvi del repartiment entre franges (2024-25)", f"{len(REGIM_B)} dies", "R4 només amb dies del règim A (principal) i amb variable de règim (secundari); sense 2025"],
    ["3", "Forat de cobertura dels serveis (gen.-mar. 2023)", f"{int(ANOMALIES['Serveis']['block'].sum())} dies de serveis", "exclòs pel detector de blocs; també el del residencial de principis de gener"],
    ["4", "Reclassificacions de sector (ago. 2023; set. 2025)", "serveis i indústria", "prova de composició amb el total de tots els sectors"],
    ["5", "Dies dolents i bloc d'agost del 2025", f"{int(ANOMALIES['Residencial']['single'].sum() + ANOMALIES['Residencial']['block'].sum())} dies del residencial", "detector condicionat a la temperatura; els dies extrems mai s'exclouen"],
    ["6", "Unitats: el portal diu MWh, són kWh", "—", "comprovació de l'ordre de magnitud (secció 3)"],
    ["7", "Sense validació externa del total", "—", "limitació declarada; demanar el total anual a l'Ajuntament o l'ICAEN"],
], columns=["alerta", "què", "abast", "tractament"])
print(REGISTRE.to_string(index=False))
REGISTRE.to_csv("registre_alertes_v13.csv", index=False)

## 5. Llindar de confort $T_c$

**Què fa:** prova llindars de 18 a 25 °C i tria el que minimitza l'error del model (residencial net, estius 2021-2025, sense els dies d'onada).

**Per què l'error quadràtic mitjà:** el model té un colze a $T_c$. El llindar que s'equivoca menys és l'estimació de mínims quadrats d'on és el colze.

**Què cal mirar:** que el mínim sigui clar. Si fos pla, el llindar estaria mal determinat; en aquest cas, la secció 7 comprova que el creixement no en depèn.

In [ ]:
RES = NET["Residencial"]
TC, perfil = ct.choose_tc(RES, temp, SPEC["tc_candidats"], LAGS, FESTIUS, exclude=ONADES, years=ANYS)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(perfil["tc"], perfil["mse"], "o-"); ax.axvline(TC, ls="--", color=fs.INK_2)
ax.set_xlabel("$T_c$ (°C)"); ax.set_ylabel("Error quadràtic mitjà"); ax.set_title(f"Comfort threshold: $T_c$ = {TC} °C")
plt.tight_layout(); plt.savefig("figures_v13/tria_tc.png"); plt.show()
print(f"[OK] Tc = {TC} °C · l'error varia un {100 * (perfil['mse'].max() / perfil['mse'].min() - 1):.0f} % entre llindars")

## 6. R1 · La resposta de la ciutat i el test de linealitat

**Què fa, en tres parts:**
1. **Funció de resposta** $\chi(k)$: quant consum de més hi ha $k$ dies després d'un grau extra (tots els estius junts, sense onades).
2. **Comprovació per correlacions:** la mateixa $\chi$ calculada amb les covariàncies desfasades entre consum i temperatura. En un sistema lineal amb el forçament observat ha de coincidir amb la regressió.
3. **Test de les onades, ara amb la $\chi$ de cada estiu:** el model s'ajusta sense els dies d'onada i després els prediu. Les versions anteriors feien servir la mateixa $\chi$ tots els estius, i amb la tendència això feia sortir residus positius espuris.

**Què cal mirar:** les dues estimacions de $\chi^0$ han de coincidir, i el residu d'onada ha de ser ≈ 0 (sense saturació ni amplificació).

In [ ]:
r1 = ct.fit_city(RES, temp, TC, LAGS, FESTIUS, exclude=ONADES, years=ANYS)
chi_k = np.array([100 * r1["beta"][f"cdd{k}"] for k in range(LAGS + 1)])
se_k = np.array([100 * np.sqrt(r1["cov"].loc[f"cdd{k}", f"cdd{k}"]) for k in range(LAGS + 1)])
reg, cor = ct.correlation_check(RES, temp, TC, LAGS, ANYS, FESTIUS, exclude=ONADES)
ones, ones_any, sd_ones = ct.heatwave_residuals(RES, temp, TC, ONADES, LAGS, ANYS, FESTIUS)

print(f"χ⁰ (tots els estius) = {r1['chi0']:.2f} ± {r1['se']:.2f} %/K · {r1['n']} dies")
print("χ(k) %/K:", np.round(chi_k, 2), " · fracció el mateix dia:", f"{chi_k[0] / chi_k.sum():.0%}")
print(f"Comprovació per correlacions: χ⁰ regressió {reg.sum():.2f} vs correlacions {cor.sum():.2f} %/K")
print(f"\nTest de les onades (χ de cada estiu): residu mitjà {ones['residu_pct'].mean():+.1f} % "
      f"en {len(ones)} dies (dispersió dels dies normals ±{sd_ones:.1f} %)")
print(ones_any.round(1).to_string())

fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
axs[0].errorbar(range(LAGS + 1), chi_k, yerr=2 * se_k, fmt="o-", color=fs.CAT[0])
axs[0].axhline(0, color=fs.INK_2, lw=0.8); axs[0].set_xticks(range(LAGS + 1))
axs[0].set_xlabel("Lag k (days)"); axs[0].set_ylabel("χ(k) (% per K)")
axs[0].set_title("a) Response function of the city")
axs[1].scatter(ones["tmean"], ones["residu_pct"], s=18, color=fs.CAT[1])
axs[1].axhline(0, color=fs.INK_2, lw=0.8); axs[1].axhspan(-2 * sd_ones, 2 * sd_ones, color=fs.GRID, zorder=0)
axs[1].set_xlabel("Daily mean temperature (°C)"); axs[1].set_ylabel("Observed − predicted (%)")
axs[1].set_title("b) Heatwave days predicted by the linear response")
plt.tight_layout(); plt.savefig("figures_v13/fig1_response_function.png"); plt.show()

## 7. R2 · La tendència i la seva robustesa

**Què fa:** estima $\chi^0$ per a cada estiu i el canvi entre el 2021 i el 2024 (principal), a més del 2022→2024 i el pendent lineal. Després repeteix el creixement canviant **una cosa cada vegada**:

| Prova | Què descarta |
|---|---|
| Tots els sectors sumats | Reclassificacions de sector i canvis de cobertura (alertes 3 i 4) |
| Resposta absoluta (MWh/dia per K) | Que només hagi baixat el consum de base |
| $T_c$ de 18 a 23 °C | Que depengui del llindar |
| Amb els dies d'onada | La selecció de dies |
| Terme quadràtic comú | Una resposta corbada |
| Punt de rosada (control i interacció) | La humitat |
| Preu (PVPC) i interacció amb la calor | El preu de l'electricitat |
| Sense el 2025 | L'any de qualitat inferior (alerta 1) |
| Només juny i setembre / només juliol i agost | **On** creix (mecanisme) |

**Què cal mirar:** que el creixement es mantingui positiu i significatiu en totes les proves excepte la de juny i setembre, on esperem que no hi sigui.

In [ ]:
def creixement(serie, tc=None, exclude=ONADES, years=ANYS, extra=None, months=None):
    kw = {} if months is None else {"months": months}
    tab, b, c = ct.response_by_year(serie, temp, TC if tc is None else tc, LAGS, years, FESTIUS,
                                    exclude=exclude, extra=extra, **kw)
    d, se = ct.growth(b, c, A0, A1)
    return tab, b, c, d, se

tab_res, b_res, c_res, d0, se0 = creixement(RES)
d22, se22 = ct.growth(b_res, c_res, 2022, A1)
pend, se_pend = ct.slope_over_years(b_res, c_res, ANYS)
print("Residencial · χ⁰ per estiu (%/K):", {a: f"{r.chi0:.2f} ± {r.se:.2f}" for a, r in tab_res.iterrows()})
print(f"Creixement {A0}→{A1}: {d0:+.2f} ± {se0:.2f} (z = {d0 / se0:.1f}) · 2022→{A1}: {d22:+.2f} ± {se22:.2f} · pendent {pend:+.2f} ± {se_pend:.2f} %/K per any")

PROVES = {"Base (residencial)": (d0, se0)}
PROVES["Tots els sectors sumats"] = creixement(NET["Total"])[3:]
for tc_ in [18.0, 19.0, 21.0, 22.0, 23.0]:
    PROVES[f"Tc = {tc_:.0f} °C"] = creixement(RES, tc=tc_)[3:]
PROVES["Amb els dies d'onada"] = creixement(RES, exclude=None)[3:]
PROVES["Terme quadràtic comú"] = creixement(RES, extra=lambda t: pd.DataFrame({"cdd_quad": t["cdd0"] ** 2}))[3:]
if humitat is not None:
    td = humitat.set_index("data")["dew_point_2m_mean"]
    def ext_hum(t):
        x = t["data"].map(td).fillna(td.mean()).values - td.mean()
        return pd.DataFrame({"rosada": x, "cdd_x_rosada": t["cdd0"].values * x})
    PROVES["Punt de rosada (control + interacció)"] = creixement(RES, extra=ext_hum)[3:]
if preus is not None:
    pv = preus.set_index("data")["PVPC"]
    def ext_preu(t):
        x = (t["data"].map(pv) - pv.mean()) / pv.std()
        x = x.fillna(0).values
        return pd.DataFrame({"preu": x, "cdd_x_preu": t["cdd0"].values * x})
    PROVES["Preu PVPC (control + interacció)"] = creixement(RES, extra=ext_preu)[3:]
PROVES["Sense el 2025"] = creixement(RES, years=[a for a in ANYS if a != 2025])[3:]
PROVES["Només juny i setembre"] = creixement(RES, months=(6, 9))[3:]
PROVES["Només juliol i agost"] = creixement(RES, months=(7, 8))[3:]
ROB = pd.DataFrame(PROVES, index=["creixement", "se"]).T
ROB["z"] = ROB["creixement"] / ROB["se"]
print("\nCreixement de χ⁰ 2021→2024 (%/K) en cada prova:"); print(ROB.round(2).to_string())

# Resposta absoluta (MWh/dia per K): residencial i total
ABS = {n: ct.absolute_response_by_year(NET[n], temp, TC, LAGS, ANYS, FESTIUS, exclude=ONADES) for n in ["Residencial", "Total"]}
print("\nResposta absoluta (MWh/dia per K):"); print(pd.DataFrame({n: a["resp"] for n, a in ABS.items()}).round(0).to_string())
tab_ser = creixement(NET["Serveis"])[0]
print("\nServeis (informatiu) · χ⁰ per estiu:", tab_ser["chi0"].round(2).to_dict())

fig, axs = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.3]})
axs[0].errorbar(tab_res.index, tab_res["chi0"], yerr=2 * tab_res["se"], fmt="o-", color=fs.CAT[0], label="Residential")
tab_tot = creixement(NET["Total"])[0]
axs[0].errorbar(tab_tot.index + 0.08, tab_tot["chi0"], yerr=2 * tab_tot["se"], fmt="s--", color=fs.CAT[1], label="All sectors")
axs[0].set_xticks(ANYS); axs[0].set_ylabel("χ⁰ (% per K)"); axs[0].set_title("a) Static response, each summer"); axs[0].legend()
y = np.arange(len(ROB))[::-1]
axs[1].errorbar(ROB["creixement"], y, xerr=2 * ROB["se"], fmt="o", color=fs.CAT[0])
EN = {"Base (residencial)": "Baseline (residential)", "Tots els sectors sumats": "All sectors summed",
      "Amb els dies d'onada": "Heatwave days included", "Terme quadràtic comú": "Common quadratic term",
      "Punt de rosada (control + interacció)": "Dew point (control + interaction)",
      "Preu PVPC (control + interacció)": "Electricity price (control + interaction)",
      "Sense el 2025": "Without 2025", "Només juny i setembre": "June and September only",
      "Només juliol i agost": "July and August only"}
axs[1].axvline(0, color=fs.INK_2, lw=0.8); axs[1].set_yticks(y)
axs[1].set_yticklabels([EN.get(n, n) for n in ROB.index], fontsize=8)
axs[1].set_xlabel(f"Change of χ⁰ {A0}→{A1} (% per K, ±2 s.e.)"); axs[1].set_title("b) Robustness of the growth")
plt.tight_layout(); plt.savefig("figures_v13/fig2_trend_robustness.png"); plt.show()

## 8. R3 · On creix: corbes sense forma imposada i model de llindars

**Què fa:**
1. Calcula, per a cada estiu, el consum en cada grau de temperatura (mitjana dels 3 últims dies) respecte dels dies ≤ 20 °C, **sense imposar cap forma** a la corba.
2. **Test pre-especificat (v0.9):** per sota de 24 °C, corbes iguals entre estius; per sobre, el **model de llindars**, és a dir, la mateixa forma × un factor λ per estiu (λ₂₀₂₁ = 1).
3. **Test de tota la corba (*post hoc*, 24/10/2026):** el model de llindars de 21 a 28 °C. El vam afegir en veure que la part de sota de 24 °C fallava perquè el 2025 ja és més alt a 22-23 °C, cosa que vol dir que el factor també actua allà.

**Lectura física:** si cada llar té un llindar d'encesa i la ciutat és la suma, la forma de la corba és la distribució de llindars i λ és la quantitat d'aparells per l'ús de cada aparell. Que la forma no canviï i λ creixi vol dir **més aparells o més ús, no un comportament diferent**. Segons el pilot dels certificats (notebook a part), els aparells nous només expliquen un 13-42 % del creixement de λ.

**Què cal mirar:** p > 0,05 vol dir que el model de llindars és compatible amb les dades. Amb les dades del 24/10, la part de sota de 24 °C del test pre-especificat falla, mentre que el model de llindars passa tant per sobre de 24 °C com a tota la corba. Per tant, la lectura és «canvi d'amplitud amb la mateixa forma», no «canvi només a la part alta».

In [ ]:
Y, S = ct.response_curves(RES, temp, ANYS, FESTIUS, bins=range(21, 29), floor=20)
baix, alt = [b for b in Y.index if b < 24], [b for b in Y.index if b >= 24]
chi2_b, dof_b = ct.curves_equal(Y.loc[baix], S.loc[baix])
lam_a, se_a, _, chi2_a, dof_a = ct.threshold_factor(Y.loc[alt], S.loc[alt], ref=A0)
lam, se_lam, g, chi2_t, dof_t = ct.threshold_factor(Y, S, ref=A0)
p_b, p_a, p_t = (1 - stats.chi2.cdf(c, d) for c, d in [(chi2_b, dof_b), (chi2_a, dof_a), (chi2_t, dof_t)])
print("Test pre-especificat:")
print(f"  < 24 °C, corbes iguals: χ² = {chi2_b:.1f} amb {dof_b} g.ll. (p = {p_b:.2f})")
print(f"  ≥ 24 °C, forma comuna × factor: χ² = {chi2_a:.1f} amb {dof_a} g.ll. (p = {p_a:.2f}) · λ = {lam_a.round(2).to_dict()}")
print("Tota la corba (post hoc):")
print(f"  21-28 °C, forma comuna × factor: χ² = {chi2_t:.1f} amb {dof_t} g.ll. (p = {p_t:.2f})")
print("  λ per estiu (2021 = 1):", {a: f"{lam[a]:.2f} ± {se_lam[a]:.2f}" for a in lam.index})

cols = fs.year_colors(ANYS)
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for a in ANYS:
    yy = Y[a].dropna()
    axs[0].errorbar(np.r_[20, yy.index] + 0.06 * (a - 2023), np.r_[0, yy.values], yerr=np.r_[0, 2 * S[a].dropna().values],
                    fmt="o-", color=cols[a], label=str(a))
axs[0].set_xlabel("Mean temperature of the last 3 days (°C; 20 = ≤20)"); axs[0].set_ylabel("Consumption above ≤20 °C (log × 100)")
axs[0].set_title("a) Response curve of each summer, no imposed shape"); axs[0].legend()
axs[1].errorbar(lam.index, lam.values, yerr=2 * se_lam.values, fmt="o-", color=fs.CAT[0])
axs[1].set_xticks(ANYS); axs[1].set_ylabel("Amplitude λ (2021 = 1)")
axs[1].set_title(f"b) Amplitude of each summer (same shape, χ²/dof = {chi2_t / dof_t:.2f})")
plt.tight_layout(); plt.savefig("figures_v13/fig3_curves_threshold_model.png"); plt.show()

## 9. R4 · Dia o nit: a quina calor respon el consum, i quina resposta creix

**Què fa:** substitueix la temperatura mitjana per dos forçaments:
- **calor de dia**: T de 12 a 18 h per sobre de 27 °C;
- **calor de nit**: T de 0 a 6 h per sobre de 20 °C;

cadascun amb el mateix dia i el dia anterior (ERA5 horari). Per a cada franja del consum (00-06, 06-12, 12-18, 18-24 h) estima la resposta a cada calor i el seu canvi entre el 2021 i el 2024.

**Alerta 2:**
- **Principal:** només els dies del règim A (repartiment entre franges com al 2021-2023).
- **Secundari:** tots els dies amb una variable de règim.

El 2025 queda fora.

**Què cal mirar:** a la franja de 0 a 6 h, la resposta a la calor de **nit** ha de ser més gran que la de dia. Després, a **totes** les franges: el canvi de la resposta a la nit ha de ser positiu, i el de la resposta al dia, compatible amb zero.

In [ ]:
bandes = ct.band_temperatures(hor, {"nit": (0, 6), "dia": (12, 18)})
FORC = ct.day_night_forcings(bandes, SPEC["tc_dia"], SPEC["tc_nit"])
AF = SPEC["anys_franges"]
files = []
for tr in FRANGES:
    s = ct.city_series(df, sector="Residencial", band=tr, days=DIES_BONS["Residencial"])
    for versio, kw in [("règim A (principal)", dict(exclude=REGIM_B)), ("variable de règim", dict(regime_b=REGIM_B))]:
        b, c = ct.day_night_response(s, temp, FORC, TC, LAGS, AF, FESTIUS, **kw)
        bb, cc = ct.day_night_response(s, temp, FORC, TC, LAGS, AF, FESTIUS, by_year=True, **kw)
        files.append({"franja": NOM_F[tr], "versió": versio, **ct.day_night_summary(b, c),
                      **{k: v for k, v in ct.day_night_summary(bb, cc, True, A0, A1).items() if "canvi" in k}})
DN = pd.DataFrame(files).set_index(["franja", "versió"])
print("Resposta (%/K) a la calor de dia i de nit, i canvi 2021→2024:"); print(DN.round(2).to_string())

princ = DN.xs("règim A (principal)", level="versió")
x = np.arange(len(princ))
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for i, (nom, lab) in enumerate([("dia", "Day heat (12-18 h)"), ("nit", "Night heat (0-6 h)")]):
    axs[0].bar(x + 0.38 * i, princ[nom], 0.36, yerr=2 * princ[f"se_{nom}"], color=fs.CAT[i], label=lab, error_kw={"ecolor": fs.INK_2, "elinewidth": 1})
    axs[1].bar(x + 0.38 * i, princ[f"canvi_{nom}"], 0.36, yerr=2 * princ[f"se_canvi_{nom}"], color=fs.CAT[i], label=lab, error_kw={"ecolor": fs.INK_2, "elinewidth": 1})
for ax, t in zip(axs, ["a) Response of each time band", f"b) Change {A0}→{A1}"]):
    ax.set_xticks(x + 0.19); ax.set_xticklabels(princ.index); ax.axhline(0, color=fs.INK_2, lw=0.8)
    ax.set_xlabel("Consumption time band (h)"); ax.set_ylabel("% per K"); ax.set_title(t); ax.legend()
plt.tight_layout(); plt.savefig("figures_v13/fig4_day_night.png"); plt.show()

## 10. R5 · El fred i la simetria de la V

**Què fa:** el model d'hivern amb graus-dia de calefacció $\mathrm{HDD}=\max(T_h-T,0)$ i efectes fixos d'hivern, setmana ISO (absorbeix el cicle de la llum), dia, festiu i Nadal. Tria $T_h$ igual que $T_c$. Estima $\chi_h$ per hivern de tres maneres:
- **novembre-març**, l'especificació inicial;
- **desembre-febrer**, *post hoc*;
- **novembre-març amb efectes fixos d'hivern × mes**, que absorbeix salts de nivell.

Després calcula $R=\chi_h/\chi_c$.

**Què cal mirar:** si el canvi de $\chi_h$ és robust a les tres versions. El 24/10 no ho era, i el resultat que sí ho era és el **nivell** de $\chi_h$ i la caiguda de $R$.

**Compte:** el gas no el veiem. $\chi_h$ només mesura la part elèctrica de la calefacció.

In [ ]:
TH, perfil_h = ct.choose_th(RES, temp, SPEC["th_candidats"], LAGS, FESTIUS, SPEC["hiverns"])
var_h = perfil_h["mse"].max() / perfil_h["mse"].min() - 1
print(f"Th = {TH} °C (l'error varia un {100 * var_h:.0f} % entre llindars)")
H = {}
for nom, kw in [("nov.-març (inicial)", dict(months=SPEC["mesos_hivern"])),
                ("des.-feb. (post hoc)", dict(months=SPEC["mesos_hivern_cor"])),
                ("nov.-març + hivern × mes", dict(months=SPEC["mesos_hivern"], winter_month=True))]:
    tab, b, c = ct.response_by_winter(RES, temp, TH, LAGS, SPEC["hiverns"], FESTIUS, **kw)
    d, se = ct.growth(b, c, SPEC["hiverns"][0], SPEC["hiverns"][-1], LAGS, prefix="hdd")
    H[nom] = tab.assign(canvi=d, se_canvi=se)
    print(f"  {nom:26s} χ_h: " + " / ".join(f"{v:.2f}" for v in tab["chi_h"]) + f"  · canvi {d:+.2f} ± {se:.2f}")
pool = ct.fit_winter(RES, temp, TH, LAGS, FESTIUS, SPEC["hiverns"])
print(f"Nivell (tots els hiverns, nov.-març): χ_h = {pool['chi_h']:.2f} ± {pool['se']:.2f} %/K")

SIM = []
for nom, tab in H.items():
    for w in SPEC["hiverns"]:
        Rv, sR = ct.asymmetry(tab.loc[w, "chi_h"], tab.loc[w, "se"], tab_res.loc[w, "chi0"], tab_res.loc[w, "se"])
        SIM.append({"versió": nom, "any": w, "R": Rv, "se": sR})
SIM = pd.DataFrame(SIM)
print("\nR = χ_h (hivern que comença el nov. de l'any) / χ_c (estiu de l'any):")
print(SIM.pivot(index="any", columns="versió", values="R").round(2).to_string())

EN_H = {"nov.-març (inicial)": "Nov-Mar (pre-specified)", "des.-feb. (post hoc)": "Dec-Feb (post hoc)",
        "nov.-març + hivern × mes": "Nov-Mar + winter × month effects"}
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for i, (nom, tab) in enumerate(H.items()):
    axs[0].errorbar(np.array(tab.index) + 0.06 * i, tab["chi_h"], yerr=2 * tab["se"], fmt="o-", color=fs.CAT[i], label=EN_H[nom])
    g_ = SIM[SIM["versió"] == nom]
    axs[1].errorbar(g_["any"] + 0.06 * i, g_["R"], yerr=2 * g_["se"], fmt="o-", color=fs.CAT[i], label=EN_H[nom])
axs[0].set_ylabel(r"$\chi_h$ (% per K below $T_h$)"); axs[0].set_title("a) Response to cold, each winter (starting year)")
axs[1].set_ylabel(r"$R = \chi_h / \chi_c$"); axs[1].set_title("b) Symmetry of the V")
for ax in axs: ax.set_xticks(SPEC["hiverns"]); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("figures_v13/fig5_cold_symmetry.png"); plt.show()

## 11. R6 · Histèresi: test de fronts

**Què fa:** comprova si, a la mateixa temperatura, el consum és més alt quan la calor se'n va que quan arriba, un cop descomptada la memòria. El terme $\gamma\,\mathrm{CDD}(d)\,\mathbf 1[T(d)<T(d-1)]$ s'afegeix a una memòria additiva flexible (frontisses per retard). Una memòria lineal o corbada dona $\gamma = 0$; la histèresi de relés (aire encès a 25 °C i apagat a 23 °C) dona $\gamma > 0$.

**Calibratge amb dades sintètiques:** falsos positius del 0-3 %; potència del 100 % amb una histèresi d'1 K, del 62 % amb 2 K i del 10 % amb 3 K. **Un nul vol dir «no hi ha histèresi estreta».**

**Què cal mirar:** $|z| < 2$ a totes les versions és un resultat nul robust, que confirma el model lineal (*crossover*, no transició).

In [ ]:
HIS = {}
for nom, kw in [("base", {}), ("sense onades", dict(exclude=ONADES)), (f"Tc = {TC - 1}", dict(tc=TC - 1)),
                (f"Tc = {TC + 1}", dict(tc=TC + 1)), ("estius 2021-22", dict(years=[2021, 2022])),
                ("estius 2023-24", dict(years=[2023, 2024]))]:
    kw = dict(kw); tc_ = kw.pop("tc", TC); yrs = kw.pop("years", ANYS)
    h = ct.hysteresis_test(RES, temp, tc_, LAGS, FESTIUS, SPEC["mesos_estiu"], yrs, **kw)
    HIS[nom] = {k: h[k] for k in ["gamma", "se", "z", "n"]}
HIS = pd.DataFrame(HIS).T
print("γ (% per K de més els dies que la temperatura baixa):"); print(HIS.round(3).to_string())

## 12. Calor antropogènica: sensibilitat mesurada i ordre de magnitud del bucle

**Què fa:** converteix la resposta absoluta de **tots els sectors** (MWh/dia per K, secció 7) en un flux per unitat d'àrea:
$$\frac{\partial Q_F}{\partial T} = \frac{\Delta E\ [\mathrm{MWh/dia/K}] \times 10^6}{24\ \mathrm{h} \times A}\quad [\mathrm{W\,m^{-2}\,K^{-1}}]$$
amb $A$ = 101 km² (l'àrea dels 42 CP, una reconstrucció). Tota l'electricitat acaba com a calor (balanç diari). Llavors $G = (\partial Q_F/\partial T)/h_\text{eff}$.

**Compte:**
- $h_\text{eff}$ (10-50 W m⁻² K⁻¹) **no està verificat**: $G$ és només un ordre de magnitud per a la discussió, no un resultat.
- La calor es retorna sobretot de nit, quan la capa límit és baixa. Per això la $G$ local i instantània pot ser més gran.

In [ ]:
a_m2 = SPEC["area_km2"] * 1e6
QF = (ABS["Total"]["resp"] * 1e6 / 24 / a_m2).rename("dQF_dT_W_m2_K").to_frame()
QF["se"] = ABS["Total"]["se"] * 1e6 / 24 / a_m2
QF["G_min"], QF["G_max"] = QF.iloc[:, 0] / SPEC["h_eff"][1], QF.iloc[:, 0] / SPEC["h_eff"][0]
print("∂Q_F/∂T (W m⁻² K⁻¹) de tota la ciutat i G per a h_eff = 50 i 10 W m⁻² K⁻¹ (NO verificat):")
print(QF.round(3).to_string())

## 13. Resum de resultats, figures i exportació

**Què fa:** reuneix tots els números de l'article en una sola taula, desa les figures (en anglès, 300 ppp) i guarda un `.zip` al Drive perquè Claude pugui revisar-ho tot.

In [ ]:
nit = DN.loc[("00-06", "règim A (principal)")]
RESUM = pd.DataFrame([
    ["R1", "Tc (°C)", f"{TC}", ""],
    ["R1", "χ⁰ tots els estius (%/K)", f"{r1['chi0']:.2f} ± {r1['se']:.2f}", f"χ⁰ per correlacions {cor.sum():.2f}"],
    ["R1", "Residu d'onada, χ de cada estiu (%)", f"{ones['residu_pct'].mean():+.1f}", f"{len(ones)} dies; dispersió normal ±{sd_ones:.1f} %; poca potència"],
    ["R2", f"χ⁰ {A0} → {A1} (%/K)", f"{tab_res.loc[A0, 'chi0']:.2f} → {tab_res.loc[A1, 'chi0']:.2f}", ""],
    ["R2", f"Creixement {A0}→{A1} (%/K)", f"{d0:+.2f} ± {se0:.2f}", f"z = {d0 / se0:.1f}"],
    ["R2", "Creixement, tots els sectors (%/K)", "{:+.2f} ± {:.2f}".format(*PROVES["Tots els sectors sumats"]), "prova de composició"],
    ["R2", f"Resposta absoluta total {A0} → {A1} (MWh/dia/K)", f"{ABS['Total'].loc[A0, 'resp']:.0f} → {ABS['Total'].loc[A1, 'resp']:.0f}", ""],
    ["R2", "Proves de robustesa amb z > 2", f"{int((ROB['z'] > 2).sum())} de {len(ROB)}", "juny + setembre: no s'espera"],
    ["R3", "Pre-especificat: iguals < 24 °C (p) / forma × factor ≥ 24 °C (p)", f"{p_b:.2f} / {p_a:.2f}", "la part < 24 °C falla (2025)"],
    ["R3", "Forma × factor a tota la corba 21-28 °C (p)", f"{p_t:.2f}", f"post hoc · λ{A1} = {lam[A1]:.2f} ± {se_lam[A1]:.2f}"],
    ["R4", "0-6 h: resposta a la calor de nit / de dia (%/K)", f"{nit['nit']:.2f} / {nit['dia']:.2f}", "règim A"],
    ["R4", "0-6 h: canvi de la resposta a la nit / al dia", f"{nit['canvi_nit']:+.2f} ± {nit['se_canvi_nit']:.2f} / {nit['canvi_dia']:+.2f} ± {nit['se_canvi_dia']:.2f}", ""],
    ["R4", "Franges amb canvi a la nit z > 2 / al dia z > 2",
     f"{int((princ['canvi_nit'] / princ['se_canvi_nit'] > 2).sum())} / {int((princ['canvi_dia'] / princ['se_canvi_dia'] > 2).sum())} de {len(princ)}", "règim A"],
    ["R5", "χ_h nivell (%/K), Th", f"{pool['chi_h']:.2f} ± {pool['se']:.2f}, {TH} °C", ""],
    ["R5", "Canvi de χ_h (3 versions)", " / ".join(f"{t['canvi'].iloc[0]:+.2f}" for t in H.values()), "robust només si les tres coincideixen"],
    ["R6", "γ histèresi (base)", f"{HIS.loc['base', 'gamma']:+.3f} ± {HIS.loc['base', 'se']:.3f}", f"màx |z| = {HIS['z'].abs().max():.1f}"],
    ["—", f"∂Q_F/∂T total {A1} (W m⁻² K⁻¹)", f"{QF.iloc[:, 0].loc[A1]:.2f}", "G no verificat"],
], columns=["resultat", "magnitud", "valor", "nota"])
print(RESUM.to_string(index=False))
RESUM.to_csv("resum_article1_v13.csv", index=False)

# Exportació (també al Drive, perquè la descàrrega de Colab de vegades no surt)
SORT = "dades_per_claude_v13"; os.makedirs(SORT, exist_ok=True)
for nom, s in SERIES.items(): s.to_csv(f"{SORT}/serie_{nom.lower()}.csv", index=False)
pd.Series(sorted(set().union(*[set(SERIES[n]['data']) - DIES_BONS[n] for n in SERIES])), name="data").to_csv(f"{SORT}/dies_exclosos.csv", index=False)
df.groupby(["data", "sector", "tram"], as_index=False)["kwh"].sum().to_csv(f"{SORT}/ciutat_sector_franja.csv.gz", index=False, compression="gzip")
temp.to_csv(f"{SORT}/temp_era5.csv", index=False); bandes.to_csv(f"{SORT}/temp_franges.csv", index=False)
pd.Series(sorted(REGIM_B), name="data").to_csv(f"{SORT}/regim_b.csv", index=False)
for f in ["resum_article1_v13.csv", "registre_alertes_v13.csv", "dies_exclosos_v13.csv"]: shutil.copy(f, SORT)
json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in SPEC.items()} | {"TC": TC, "TH": TH},
          open(f"{SORT}/especificacio.json", "w"), ensure_ascii=False, default=float)
with zipfile.ZipFile("dades_per_claude_v13.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(SORT): z.write(f"{SORT}/{f}", f)
    for f in os.listdir("figures_v13"): z.write(f"figures_v13/{f}", f"figures/{f}")
try:
    shutil.copy("dades_per_claude_v13.zip", f"{CARPETA}/dades_per_claude_v13.zip")
    print(f"\n[OK] dades_per_claude_v13.zip ({os.path.getsize('dades_per_claude_v13.zip') / 1e6:.1f} MB) desat a {CARPETA}: adjunta'l al xat")
except Exception as e:
    print(f"[!] No s'ha pogut desar al Drive ({e}): baixa'l del panell de fitxers de Colab")